# Import Modules

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from utils import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

In [ ]:
# 3 Datasets
df_chd_heart_stroke_intersection = get_unscaled_combined("chd-heart-stroke-intersection")
df_chd_heart_stroke_union = get_unscaled_combined("chd-heart-stroke-union")
df_framingham_heart_stroke_intersection = get_unscaled_combined("framingham-heart-stroke-intersection")
df_framingham_heart_stroke_union = get_unscaled_combined("framingham-heart-stroke-union")

# 2 Datasets
df_chd_stroke_intersection = get_unscaled_combined("chd-stroke-intersection")
df_chd_stroke_union = get_unscaled_combined("chd-stroke-union")
df_framingham_heart_intersection = get_unscaled_combined("framingham-heart-intersection")
df_framingham_heart_union = get_unscaled_combined("framingham-heart-union")

In [ ]:
union_datasets = {
    "framingham_heart_union": df_framingham_heart_union,
    "chd_stroke_union": df_chd_stroke_union,
    "chd_heart_stroke_union": df_chd_heart_stroke_union,
    "framingham_heart_stroke_union": df_framingham_heart_stroke_union
}

intersection_datasets = {
    "chd_stroke_intersection": df_chd_stroke_intersection,
    "framingham_heart_intersection": df_framingham_heart_intersection,
    "chd_heart_stroke_intersection": df_chd_heart_stroke_intersection,
    "framingham_heart_stroke_intersection": df_framingham_heart_stroke_intersection
}

In [ ]:
def check_missing_values(df, dataset_name="Dataset"):
    print(f"=== Missing Value Report: {dataset_name} ===")
    
    # 1. Check if ANY missing values exist overall
    has_missing = df.isnull().values.any()

    if has_missing:
        # 2. Count missing values for each column
        missing_counts = df.isnull().sum()
        
        # Filter to show only columns that actually have missing data
        missing_columns = missing_counts[missing_counts > 0]
        
        # 3. Calculate the percentage of missing data per column
        missing_percentages = (missing_columns / len(df)) * 100
        
        # Combine counts and percentages into a clean table
        summary_df = pd.DataFrame({
            'Missing Count': missing_columns,
            'Percentage (%)': missing_percentages.round(2)
        }).sort_values(by='Missing Count', ascending=False)
        
        print("Columns with missing values:")
        print(summary_df)
        print("\n")
    else:
        print("Great news! This dataset is completely full with no missing values.\n")


check_missing_values(df_framingham_heart_union, "Framingham-Heart Union")
check_missing_values(df_chd_stroke_union, "CHD-Stroke Union")
check_missing_values(df_chd_heart_stroke_union, "CHD-Heart-Stroke Union")
check_missing_values(df_framingham_heart_stroke_union, "Framingham-Heart-Stroke Union")
check_missing_values(df_chd_stroke_intersection, "CHD-Stroke Intersection")
check_missing_values(df_framingham_heart_intersection, "Framingham-Heart Intersection")
check_missing_values(df_chd_heart_stroke_intersection, "CHD-Heart-Stroke Intersection")
check_missing_values(df_framingham_heart_stroke_intersection, "Framingham-Heart-Stroke Intersection")

# Data Preprocessing

In [ ]:
# Define known categorical columns per dataset
CATEGORICAL_COLS = {
    'framingham_heart_union': ['edu', 'smoking_status', 'bp_status', 'stroke_status', 
                                'hypertension_status', 'diabetes_status', 'sex'],
    'chd_stroke_union': ['sex', 'hypertension_status', 'heart_disease_status', 
                          'marital_status', 'work_type', 'residence_type', 'smoking_status'],
    'chd_heart_stroke_union': ['sex', 'hypertension_status', 'heart_disease_status', 
                                'marital_status', 'work_type', 'residence_type', 'smoking_status'],
    'framingham_heart_stroke_union': ['edu', 'smoking_status', 'bp_status', 'stroke_status',
                                       'hypertension_status', 'diabetes_status', 
                                       'heart_disease_status', 'marital_status', 
                                       'work_type', 'residence_type', 'sex'],
}

def process_union(df, name, drop_threshold=0.75):
    df_processed = df.copy()

    for col in df_processed.columns:
        try:
            df_processed[col] = pd.to_numeric(df_processed[col], errors='raise')
        except (ValueError, TypeError):
            pass

    missing_fractions = df_processed.isnull().mean()
    cols_to_drop = missing_fractions[missing_fractions > drop_threshold].index
    if len(cols_to_drop) > 0:
        print(f"⚠️ [{name}] Dropping columns (> {drop_threshold*100}% missing): {list(cols_to_drop)}")
        df_processed.drop(columns=cols_to_drop, inplace=True)

    known_cats = CATEGORICAL_COLS.get(name, [])
    categorical_cols = [col for col in known_cats if col in df_processed.columns]
    numeric_cols = [col for col in df_processed.columns 
                    if col not in categorical_cols and col != 'target']

    print(f"📊 [{name}] Numeric cols: {numeric_cols}")
    print(f"📝 [{name}] Categorical cols: {categorical_cols}")

    if len(numeric_cols) > 0:
        num_imputer = SimpleImputer(strategy='median')
        df_processed[numeric_cols] = num_imputer.fit_transform(df_processed[numeric_cols])

    if len(categorical_cols) > 0:
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df_processed[categorical_cols] = cat_imputer.fit_transform(df_processed[categorical_cols])

    scaler = MinMaxScaler()
    if len(numeric_cols) > 0:
        df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

    set_preprocessed_combined(df_processed, scaler, name)
    print(f"✅ Processed Union Dataset: {name}\n")

print("--- Processing Union Datasets ---")
for name, df in union_datasets.items():
    process_union(df, name)

In [ ]:
def process_intersection(df, name):
    df_processed = df.copy()
    numeric_cols = df_processed.select_dtypes(include=['number']).columns

    scaler = MinMaxScaler()
    if len(numeric_cols) > 0:
        df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

    set_preprocessed_combined(df_processed, scaler, name)
    print(f"✅ Processed Intersection Dataset: {name}")
    
print("\n--- Processing Intersection Datasets ---")
for name, df in intersection_datasets.items():
    process_intersection(df, name)